In [1]:
import ipywidgets as widgets

import geopandas as gpd

from lets_plot.geo_data import *
from lets_plot import *
LetsPlot.setup_html()

The geodata is provided by © OpenStreetMap contributors and is made available here under the Open Database License (ODbL).


In [2]:
LEVELS = ['country', 'state', 'county', 'city']
w = {
    'country': widgets.Dropdown(
        options=[''] + geocode(level='country').get_geocodes().sort_values('found name')['found name'].to_list(),
        value='',
        description='Country:',
    ),
    'state': widgets.Dropdown(
        options=[],
        description='State:',
    ),
    'county': widgets.Dropdown(
        options=[],
        description='County:',
    )
}
out = widgets.Output()

def on_change(change):
    if change['type'] != 'change' or change['name'] != 'value':
        return
    try:
        b_gdf = gpd.GeoDataFrame()
        p_gdf = gpd.GeoDataFrame()
        for i, level in enumerate(LEVELS[:-1]):
            scope = w[LEVELS[i-1]].value if i > 0 and w[LEVELS[i-1]].value else None
            if change.owner.description == '{0}:'.format(level.title()) and w[level].value:
                for j in range(len(LEVELS) - 2, i, -1):
                    w[LEVELS[j]].options = []
                if change['new'] == '':
                    b_gdf = geocode(level=level, scope=scope).inc_res(4).get_boundaries()
                else:
                    b_gdf = geocode(level=LEVELS[i+1], scope=w[level].value).inc_res(4).get_boundaries()
                    if LEVELS[i+1] in w.keys():
                        w[LEVELS[i+1]].options = [''] + b_gdf.sort_values('found name')['found name'].to_list()
                    if b_gdf.empty:
                        b_gdf = geocode(level=level, names=[w[level].value], scope=scope)\
                                .allow_ambiguous().inc_res(4).get_boundaries()
                    if level == 'county':
                        p_gdf = geocode(level='city', scope=w['county'].value).get_centroids()
                break
        if not b_gdf.empty:
            p = None
            if p_gdf.empty:
                p = ggplot() + \
                    geom_map(data=b_gdf, fill='black', color='white', tooltips=layer_tooltips().line('@{found name}'))
            else:
                p = ggplot() + \
                    geom_map(data=b_gdf, fill='black', color='white') + \
                    geom_point(data=p_gdf, shape=1, color='white', tooltips=layer_tooltips().line('@{found name}'))
            out.outputs = ()
            out.append_display_data(p)
    except:
        out.outputs = ()
        out.append_stdout('Something went wrong')

w['country'].observe(on_change)
w['state'].observe(on_change)
w['county'].observe(on_change)

display(w['country'], w['state'], w['county'], out)

Dropdown(description='Country:', options=('', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argenti…

Dropdown(description='State:', options=(), value=None)

Dropdown(description='County:', options=(), value=None)

Output()